In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Teacher Model

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda', index=1)

In [28]:
from train_eval_func import eval_loop

In [29]:
eval_loop(teacher_model, val_dataloader, task_name, device)

({'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624},
 0.8802372308877798)

# 5. Layer Score Importance

In [30]:
from structure_prune_helper import compute_layer_importance_scores, register_importance_masks, remove_importance_masks

In [31]:
register_importance_masks(teacher_model, device, register_mha_layer=True, register_mlp_layer=True,
                         register_mha=True, register_mlp=True)

  Layer 0: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 1: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 2: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 3: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 4: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 5: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 6: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 7: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 8: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 9: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 10: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 11: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [32]:
mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
    teacher_model, train_dataloader, device,
    compute_mha=True, compute_ffn=True,
    compute_heads=True, compute_neurons=True
)

Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:11<00:00,  9.63it/s]


Layer Importance Summary (separate):
          Layer     MHA(layer)     FFN(layer)    Heads(mean)  Neurons(mean)
  ---------------------------------------------------------------------------
  0                     0.7879         0.9843         0.0000         0.0000
  1                     0.7108         0.4983         0.0000         0.0000
  2                     0.5232         0.6907         0.0000         0.0000
  3                     0.4300         0.5575         0.0000         0.0000
  4                     0.4305         1.0000         0.0000         0.0000
  5                     1.0000         0.5937         0.0000         0.0000
  6                     0.2730         0.4748         0.0000         0.0000
  7                     0.2159         0.2461         0.0000         0.0000
  8                     0.3048         0.4255         0.0000         0.0000
  9                     0.2532         0.2987         0.0000         0.0000
  10                    0.1479         0.1349   

In [33]:
layer_scores = []
for idx in range(0, 24, 2):
    layer = int(idx / 2)
    layer_score = (sublayer_imp[idx] + sublayer_imp[idx+1]) / 2
    print(layer, layer_score)
    layer_scores.append(layer_score)

0 0.8548370079721432
1 0.5887043064120773
2 0.585039680877147
3 0.4760704818221425
4 0.6834934591444725
5 0.7780108213973873
6 0.3588173180990346
7 0.22315765120006437
8 0.35161277226755316
9 0.266488999862323
10 0.13714492726932062
11 0.11579482304911151


In [34]:
print(layer_scores)

[0.8548370079721432, 0.5887043064120773, 0.585039680877147, 0.4760704818221425, 0.6834934591444725, 0.7780108213973873, 0.3588173180990346, 0.22315765120006437, 0.35161277226755316, 0.266488999862323, 0.13714492726932062, 0.11579482304911151]


# 6. Calculating Similarity 

In [35]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [36]:
reps_similarity, atts_similarity, sub_similarity = cka_eval.subLayer_interleaved_pairwise(
    teacher_model, 
    train_dataloader, 
    device, 
    only_cls_token=True, 
    max_iter=115
)

CKA Sublayer Evaluation: 100%|██████████| 115/115 [00:32<00:00,  3.57it/s]


In [37]:
reps_stats = cka_eval.similarity_stats(reps_similarity)
atts_stats = cka_eval.similarity_stats(atts_similarity)
subs_stats = cka_eval.similarity_stats(sub_similarity)

print("  Reps Similarity Stats:")
print(f"    Average: {reps_stats['average']:.6f}")
print(f"    Highest: {reps_stats['max']:.6f}")
print(f"    Lowest:  {reps_stats['min']:.6f}")
print("")

print("  Atts Similarity Stats:")
print(f"    Average: {atts_stats['average']:.6f}")
print(f"    Highest: {atts_stats['max']:.6f}")
print(f"    Lowest:  {atts_stats['min']:.6f}")

print("  Sublayer Similarity Stats:")
print(f"    Average: {subs_stats['average']:.6f}")
print(f"    Highest: {subs_stats['max']:.6f}")
print(f"    Lowest:  {subs_stats['min']:.6f}")

  Reps Similarity Stats:
    Average: 0.899861
    Highest: 0.993713
    Lowest:  0.724249

  Atts Similarity Stats:
    Average: 0.944353
    Highest: 0.991281
    Lowest:  0.763968
  Sublayer Similarity Stats:
    Average: 0.436442
    Highest: 0.724606
    Lowest:  0.141554


# 7. Creating Student Model

### Layer Composition

In [38]:
layer_composition = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]

In [39]:
while len(layer_composition) > 6:
    # find high similarity pairs
    high_sim_pairs = []
    for idx in range(0, len(layer_composition) - 1):
        layer1_idx = min(layer_composition[idx])
        layer2_idx = max(layer_composition[idx + 1])
        # print(layer1_idx, layer2_idx)
        
        similarity_score = reps_similarity[layer1_idx][layer2_idx]
        if similarity_score > 0.9:
            high_sim_pairs.append((layer1_idx, layer2_idx))
    # print(high_sim_pairs)
    if len(high_sim_pairs) == 0:
        break

    # find weakest pair
    weakest_pair_idx = 0
    pair1, pair2 = high_sim_pairs[weakest_pair_idx]
    weakest_score = (layer_scores[pair1] + layer_scores[pair2]) / 2
    for idx, pair in enumerate(high_sim_pairs):
        pair1, pair2 = pair
        pair_score = (layer_scores[pair1] + layer_scores[pair2]) / 2
        if pair_score < weakest_score:
            weakest_pair_idx = idx
            weakest_score = pair_score
    # print("Weakest pair:", high_sim_pairs[weakest_pair_idx])
    # print("Idx in List:", weakest_pair_idx)
    # print("Score:", weakest_score)

    # identify in indices in layer_composition for merging
    pair1, pair2 = high_sim_pairs[weakest_pair_idx]
    for idx, layers in enumerate(layer_composition):
        if pair1 in layers:
            idx1 = idx
        elif pair2 in layers:
            idx2 = idx

    layer_composition[idx1] = layer_composition[idx1] + layer_composition[idx2]
    layer_composition[idx1].sort()
    del layer_composition[idx2]

    # update merged layer score
    highest_mha_score = 0
    highest_mlp_score  = 0
    for layer in layer_composition[idx1]:
        # print(layer)
        sublayer_idx = layer * 2
        # print(sublayer_imp[sublayer_idx], sublayer_imp[sublayer_idx+1])
    
        if sublayer_imp[sublayer_idx] > highest_mha_score:
            highest_mha_score = sublayer_imp[sublayer_idx]
    
        if sublayer_imp[sublayer_idx+1] > highest_mlp_score:
            highest_mlp_score = sublayer_imp[sublayer_idx+1]

    layer_score = (highest_mha_score + highest_mlp_score) / 2
    for layer in layer_composition[idx1]:
        layer_scores[layer] = layer_score

    # if len(layer_composition) == 8:
    #     break

In [40]:
layer_composition, len(layer_composition)

([[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]], 6)

In [41]:
print(layer_scores)

[0.8548370079721432, 0.6788142235308011, 0.6788142235308011, 0.4760704818221425, 0.6834934591444725, 0.7780108213973873, 0.7780108213973873, 0.7780108213973873, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316]


### Merging Layers

In [42]:
def merge_mha_cluster(model, layers, target_idx, k_per_layer, device):
    assert len(layers) == len(k_per_layer), "layers and k_per_layer must be same length"
    assert 0 <= target_idx < len(layers), "target_idx out of range"

    targetLayer = layers[target_idx]
    head_size   = targetLayer.attention.self.attention_head_size

    # clamp each k to the number of heads actually available in that layer
    k_per_layer_clamped = []
    for k, layer in zip(k_per_layer, layers):
        avail = layer.attention.self.num_attention_heads
        k_clamped = min(k, avail)
        if k_clamped != k:
            print(f"  WARNING: requested k={k} but layer only has {avail} heads, clamped to {k_clamped}")
        k_per_layer_clamped.append(k_clamped)
    k_per_layer = k_per_layer_clamped

    n_head  = sum(k_per_layer)
    new_dim = n_head * head_size

    print(f"Merging cluster of {len(layers)} layers into target (idx={target_idx})")
    print(f"  k_per_layer (clamped): {k_per_layer}")
    print(f"  n_head total: {n_head}, new_dim: {new_dim}")

    for i, (layer, k) in enumerate(zip(layers, k_per_layer)):
        tag = " (target)" if i == target_idx else ""
        print(f"  Layer{tag}: taking {k} heads, head_size={layer.attention.self.attention_head_size}")

    # sanity check — all layers must have the SAME head_size
    head_sizes = set(layer.attention.self.attention_head_size for layer in layers)
    if len(head_sizes) > 1:
        raise ValueError(f"All layers must have the same head_size, got {head_sizes}")

    t_keep_len = k_per_layer[target_idx] * head_size

    orig_q   = targetLayer.attention.self.query.weight.data[:t_keep_len].clone()
    orig_q_b = targetLayer.attention.self.query.bias.data[:t_keep_len].clone()
    orig_k   = targetLayer.attention.self.key.weight.data[:t_keep_len].clone()
    orig_k_b = targetLayer.attention.self.key.bias.data[:t_keep_len].clone()
    orig_v   = targetLayer.attention.self.value.weight.data[:t_keep_len].clone()
    orig_v_b = targetLayer.attention.self.value.bias.data[:t_keep_len].clone()
    orig_o   = targetLayer.attention.output.dense.weight.data[:, :t_keep_len].clone()

    targetLayer.attention.self.query   = resize_qkv(targetLayer.attention.self.query, new_dim)
    targetLayer.attention.self.key     = resize_qkv(targetLayer.attention.self.key, new_dim)
    targetLayer.attention.self.value   = resize_qkv(targetLayer.attention.self.value, new_dim)
    targetLayer.attention.output.dense = resize_output(targetLayer.attention.output.dense, new_dim)

    with torch.no_grad():
        offset = 0

        for i, (layer, k) in enumerate(zip(layers, k_per_layer)):
            write_len = k * head_size

            # guard: catch overflow BEFORE it crashes with a confusing message
            if offset + write_len > new_dim:
                raise RuntimeError(
                    f"Offset overflow at layer {i}: offset={offset}, write_len={write_len}, "
                    f"new_dim={new_dim}. Sum of k_per_layer*head_size exceeds new_dim — "
                    f"check k_per_layer values."
                )

            if i == target_idx:
                targetLayer.attention.self.query.weight.data[offset:offset+write_len] = orig_q
                targetLayer.attention.self.query.bias.data[offset:offset+write_len]   = orig_q_b
                targetLayer.attention.self.key.weight.data[offset:offset+write_len]   = orig_k
                targetLayer.attention.self.key.bias.data[offset:offset+write_len]     = orig_k_b
                targetLayer.attention.self.value.weight.data[offset:offset+write_len] = orig_v
                targetLayer.attention.self.value.bias.data[offset:offset+write_len]   = orig_v_b
                targetLayer.attention.output.dense.weight.data[:, offset:offset+write_len] = orig_o
            else:
                # guard: source layer must actually have write_len rows available
                avail_rows = layer.attention.self.query.weight.data.shape[0]
                if write_len > avail_rows:
                    raise RuntimeError(
                        f"Layer {i}: trying to write {write_len} rows but source only has "
                        f"{avail_rows} rows (num_heads={layer.attention.self.num_attention_heads})"
                    )

                targetLayer.attention.self.query.weight.data[offset:offset+write_len] = \
                    layer.attention.self.query.weight.data[:write_len]
                targetLayer.attention.self.query.bias.data[offset:offset+write_len] = \
                    layer.attention.self.query.bias.data[:write_len]

                targetLayer.attention.self.key.weight.data[offset:offset+write_len] = \
                    layer.attention.self.key.weight.data[:write_len]
                targetLayer.attention.self.key.bias.data[offset:offset+write_len] = \
                    layer.attention.self.key.bias.data[:write_len]

                targetLayer.attention.self.value.weight.data[offset:offset+write_len] = \
                    layer.attention.self.value.weight.data[:write_len]
                targetLayer.attention.self.value.bias.data[offset:offset+write_len] = \
                    layer.attention.self.value.bias.data[:write_len]

                targetLayer.attention.output.dense.weight.data[:, offset:offset+write_len] = \
                    layer.attention.output.dense.weight.data[:, :write_len]

            print(f"    Wrote {k} heads from layer {i}{' (target)' if i==target_idx else ''} "
                  f"into slots [{offset}:{offset+write_len}]")
            offset += write_len

    targetLayer.attention.self.num_attention_heads = n_head
    targetLayer.attention.self.all_head_size        = new_dim

    targetLayer.head_mask_param = nn.Parameter(
        torch.ones(n_head, dtype=torch.float32, device=device),
        requires_grad=True
    )

    print(f"  Done: target now has {n_head} heads (dim={new_dim})")
    return targetLayer

In [43]:
import torch.nn as nn

def resize_qkv(linear_layer, new_out_dim):
    new_layer = nn.Linear(
        in_features=linear_layer.in_features,
        out_features=new_out_dim,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer

In [44]:
def resize_output(linear_layer, new_in_dim):
    new_layer = nn.Linear(
        in_features=new_in_dim,
        out_features=linear_layer.out_features,
        bias=True
    ).to(linear_layer.weight.device)

    return new_layer


In [45]:
import copy

def test_merge_mha_cluster(model, layer_indices, target_idx, k_per_layer, device):
    """
    Verifies merge_mha_cluster correctness by checking that:
    1. Target's own (kept) weights are preserved exactly
    2. Each source layer's top-k weights are copied exactly into the right offset
    3. Final shapes match expectations
    4. Non-participating layers in the model are untouched
    """
    # work on a deep copy so we don't corrupt the real model
    test_model = copy.deepcopy(model)

    layers = [test_model.bert.encoder.layer[idx] for idx in layer_indices]
    head_size = layers[0].attention.self.attention_head_size

    # --- snapshot BEFORE merge ---
    before = {}
    for i, layer in enumerate(layers):
        before[i] = {
            'q_w': layer.attention.self.query.weight.data.clone(),
            'q_b': layer.attention.self.query.bias.data.clone(),
            'k_w': layer.attention.self.key.weight.data.clone(),
            'k_b': layer.attention.self.key.bias.data.clone(),
            'v_w': layer.attention.self.value.weight.data.clone(),
            'v_b': layer.attention.self.value.bias.data.clone(),
            'o_w': layer.attention.output.dense.weight.data.clone(),
            'num_heads': layer.attention.self.num_attention_heads,
        }

    # snapshot an UNRELATED layer to confirm it's untouched
    untouched_idx = next(
        i for i in range(len(test_model.bert.encoder.layer))
        if i not in layer_indices
    )
    untouched_before = test_model.bert.encoder.layer[untouched_idx].attention.self.query.weight.data.clone()

    # --- run merge ---
    merged_layer = merge_mha_cluster(test_model, layers, target_idx, k_per_layer, device)

    # --- clamp k_per_layer same way the function does, for verification ---
    k_clamped = [min(k, before[i]['num_heads']) for i, k in enumerate(k_per_layer)]

    print("\n" + "="*60)
    print("VERIFICATION")
    print("="*60)

    all_passed = True

    # --- check 1: final shape ---
    n_head_expected = sum(k_clamped)
    actual_n_head = merged_layer.attention.self.num_attention_heads
    shape_ok = (actual_n_head == n_head_expected)
    print(f"[{'PASS' if shape_ok else 'FAIL'}] num_heads: expected={n_head_expected}, actual={actual_n_head}")
    all_passed &= shape_ok

    expected_dim = n_head_expected * head_size
    actual_dim = merged_layer.attention.self.query.weight.shape[0]
    dim_ok = (actual_dim == expected_dim)
    print(f"[{'PASS' if dim_ok else 'FAIL'}] query out_features: expected={expected_dim}, actual={actual_dim}")
    all_passed &= dim_ok

    # --- check 2: each segment matches source exactly ---
    offset = 0
    for i, k in enumerate(k_clamped):
        write_len = k * head_size
        segment_q = merged_layer.attention.self.query.weight.data[offset:offset+write_len]
        expected_q = before[i]['q_w'][:write_len]

        match = torch.allclose(segment_q, expected_q, atol=1e-7)
        tag = " (target)" if i == target_idx else ""
        print(f"[{'PASS' if match else 'FAIL'}] Layer {i}{tag} Q weights "
              f"[{offset}:{offset+write_len}] match source")
        all_passed &= match

        # also check K, V, bias for thoroughness
        segment_k = merged_layer.attention.self.key.weight.data[offset:offset+write_len]
        expected_k = before[i]['k_w'][:write_len]
        match_k = torch.allclose(segment_k, expected_k, atol=1e-7)
        print(f"[{'PASS' if match_k else 'FAIL'}] Layer {i}{tag} K weights match")
        all_passed &= match_k

        segment_v = merged_layer.attention.self.value.weight.data[offset:offset+write_len]
        expected_v = before[i]['v_w'][:write_len]
        match_v = torch.allclose(segment_v, expected_v, atol=1e-7)
        print(f"[{'PASS' if match_v else 'FAIL'}] Layer {i}{tag} V weights match")
        all_passed &= match_v

        segment_qb = merged_layer.attention.self.query.bias.data[offset:offset+write_len]
        expected_qb = before[i]['q_b'][:write_len]
        match_qb = torch.allclose(segment_qb, expected_qb, atol=1e-7)
        print(f"[{'PASS' if match_qb else 'FAIL'}] Layer {i}{tag} Q bias match")
        all_passed &= match_qb

        segment_o = merged_layer.attention.output.dense.weight.data[:, offset:offset+write_len]
        expected_o = before[i]['o_w'][:, :write_len]
        match_o = torch.allclose(segment_o, expected_o, atol=1e-7)
        print(f"[{'PASS' if match_o else 'FAIL'}] Layer {i}{tag} W_O columns match")
        all_passed &= match_o

        offset += write_len

    # --- check 3: head_mask_param shape ---
    mask_ok = (merged_layer.head_mask_param.shape[0] == n_head_expected)
    print(f"[{'PASS' if mask_ok else 'FAIL'}] head_mask_param shape: "
          f"expected={n_head_expected}, actual={merged_layer.head_mask_param.shape[0]}")
    all_passed &= mask_ok

    # --- check 4: unrelated layer untouched ---
    untouched_after = test_model.bert.encoder.layer[untouched_idx].attention.self.query.weight.data
    untouched_ok = torch.allclose(untouched_before, untouched_after)
    print(f"[{'PASS' if untouched_ok else 'FAIL'}] Unrelated layer {untouched_idx} unchanged")
    all_passed &= untouched_ok

    # --- check 5: non-target source layers unchanged (they're pruned separately, not modified here) ---
    for i, idx in enumerate(layer_indices):
        if i == target_idx:
            continue
        current_layer = test_model.bert.encoder.layer[idx]
        if current_layer.attention is not None:  # not yet pruned
            src_unchanged = torch.allclose(
                current_layer.attention.self.query.weight.data,
                before[i]['q_w']
            )
            print(f"[{'PASS' if src_unchanged else 'FAIL'}] Source layer {idx} "
                  f"itself unmodified (only target was written to)")
            all_passed &= src_unchanged

    print("="*60)
    print(f"OVERALL: {'ALL TESTS PASSED ✓' if all_passed else 'SOME TESTS FAILED ✗'}")
    print("="*60)

    return all_passed

In [46]:
import math

In [47]:
mha_mlp_contribution = {}

In [48]:
for layers in layer_composition:
    if len(layers) == 1:
        continue
    print(layers)

    head_total_imp = 0
    # Track highest MLP importance
    mlp_highest_score = -float("inf")
    mlp_highest_layer = None

    # ----------------------------
    # Calculate cluster statistics
    # ----------------------------
    for layer in layers:
        mha_score = sublayer_imp[layer * 2]
        mlp_score = sublayer_imp[layer * 2 + 1]
        
        # total MHA importance
        head_total_imp += mha_score
        
        # find strongest FFN layer
        if mlp_score > mlp_highest_score:
            mlp_highest_score = mlp_score
            mlp_highest_layer = layer
            
    print(head_total_imp)
    print("Layer with highest mlp score:", mlp_highest_layer)
    obj = {"n_heads_per_layer": {}, "mlp_highest_layer": mlp_highest_layer}

    for layer in layers:
        head_graft_n = math.ceil((12 * sublayer_imp[layer*2]) / head_total_imp)
        print(f'Layer: {layer}, Number of Heads: {head_graft_n}')
        obj['n_heads_per_layer'][layer] = head_graft_n

    mha_mlp_contribution[layer] = obj

[1, 2]
1.2339795314892967
Layer with highest mlp score: 2
Layer: 1, Number of Heads: 7
Layer: 2, Number of Heads: 6
[5, 6, 7]
1.4888206890869444
Layer with highest mlp score: 5
Layer: 5, Number of Heads: 9
Layer: 6, Number of Heads: 3
Layer: 7, Number of Heads: 2
[8, 9, 10, 11]
0.7712432476168127
Layer with highest mlp score: 8
Layer: 8, Number of Heads: 5
Layer: 9, Number of Heads: 4
Layer: 10, Number of Heads: 3
Layer: 11, Number of Heads: 2


In [49]:
mha_mlp_contribution

{2: {'n_heads_per_layer': {1: 7, 2: 6}, 'mlp_highest_layer': 2},
 7: {'n_heads_per_layer': {5: 9, 6: 3, 7: 2}, 'mlp_highest_layer': 5},
 11: {'n_heads_per_layer': {8: 5, 9: 4, 10: 3, 11: 2}, 'mlp_highest_layer': 8}}

In [50]:
layer_indices = sorted(list(mha_mlp_contribution[11]['n_heads_per_layer'].keys()))
print("Layers in the cluster:", layer_indices)

layers = []
k_per_layer = []
for layer_idx in layer_indices:
    layers.append(model.bert.encoder.layer[layer_idx])
    
    n_head = mha_mlp_contribution[11]['n_heads_per_layer'][layer_idx]
    k_per_layer.append(n_head)
print("k_per_layer:", k_per_layer)

Layers in the cluster: [8, 9, 10, 11]
k_per_layer: [5, 4, 3, 2]


In [51]:
model.bert.encoder.layer[0].attention.self.attention_head_size

64

In [52]:
merged_layer = merge_mha_cluster(model, layers, 0, k_per_layer, device)

Merging cluster of 4 layers into target (idx=0)
  k_per_layer (clamped): [5, 4, 3, 2]
  n_head total: 14, new_dim: 896
  Layer (target): taking 5 heads, head_size=64
  Layer: taking 4 heads, head_size=64
  Layer: taking 3 heads, head_size=64
  Layer: taking 2 heads, head_size=64
    Wrote 5 heads from layer 0 (target) into slots [0:320]
    Wrote 4 heads from layer 1 into slots [320:576]
    Wrote 3 heads from layer 2 into slots [576:768]
    Wrote 2 heads from layer 3 into slots [768:896]
  Done: target now has 14 heads (dim=896)


In [53]:
model.bert.encoder.layer[8].attention.self.query

Linear(in_features=768, out_features=896, bias=True)

In [54]:
model.bert.encoder.layer[8].attention.self.query.weight[320:576] == model.bert.encoder.layer[9].attention.self.query.weight[0:576 - 320]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [62]:
model.bert.encoder.layer[8].attention.self.query.weight[576:768] == model.bert.encoder.layer[10].attention.self.query.weight[0: 768 - 576]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [63]:
model.bert.encoder.layer[8].attention.self.query.weight[768:896] == model.bert.encoder.layer[11].attention.self.query.weight[0:896 - 768]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [56]:
model.bert.encoder.layer[8].attention.self.key

Linear(in_features=768, out_features=896, bias=True)

In [57]:
model.bert.encoder.layer[8].attention.self.key.weight[320:576] == model.bert.encoder.layer[9].attention.self.key.weight[0:576 - 320]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [58]:
model.bert.encoder.layer[8].attention.self.key.weight[576:768] == model.bert.encoder.layer[10].attention.self.key.weight[0: 768 - 576]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [64]:
model.bert.encoder.layer[8].attention.self.key.weight[768:896] == model.bert.encoder.layer[11].attention.self.key.weight[0:896 - 768]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [65]:
model.bert.encoder.layer[8].attention.self.value

Linear(in_features=768, out_features=896, bias=True)

In [66]:
model.bert.encoder.layer[8].attention.self.value.weight[320:576] == model.bert.encoder.layer[9].attention.self.value.weight[0:576 - 320]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [67]:
model.bert.encoder.layer[8].attention.self.value.weight[576:768] == model.bert.encoder.layer[10].attention.self.value.weight[0: 768 - 576]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [68]:
model.bert.encoder.layer[8].attention.self.value.weight[768:896] == model.bert.encoder.layer[11].attention.self.value.weight[0:896 - 768]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [73]:
model.bert.encoder.layer[8].attention.output.dense

Linear(in_features=896, out_features=768, bias=True)

In [78]:
model.bert.encoder.layer[8].attention.output.dense.weight[:,320:576] == model.bert.encoder.layer[9].attention.output.dense.weight[:,0:576 - 320]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [79]:
model.bert.encoder.layer[8].attention.output.dense.weight[:,576:768] == model.bert.encoder.layer[10].attention.output.dense.weight[:,0:768 -576]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [75]:
model.bert.encoder.layer[8].attention.output.dense.weight[:,768:896] == model.bert.encoder.layer[11].attention.output.dense.weight[:,0:896 - 768]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [69]:
test_merge_mha_cluster(
    model         = model,
    layer_indices = layer_indices,   # layers in the cluster
    target_idx    = 0,           # layer 2 is target
    k_per_layer   = k_per_layer,  # keep all 12 from target, take 4 from layer3, 2 from layer5
    device        = device
)

Merging cluster of 4 layers into target (idx=0)
  k_per_layer (clamped): [5, 4, 3, 2]
  n_head total: 14, new_dim: 896
  Layer (target): taking 5 heads, head_size=64
  Layer: taking 4 heads, head_size=64
  Layer: taking 3 heads, head_size=64
  Layer: taking 2 heads, head_size=64
    Wrote 5 heads from layer 0 (target) into slots [0:320]
    Wrote 4 heads from layer 1 into slots [320:576]
    Wrote 3 heads from layer 2 into slots [576:768]
    Wrote 2 heads from layer 3 into slots [768:896]
  Done: target now has 14 heads (dim=896)

VERIFICATION
[PASS] num_heads: expected=14, actual=14
[PASS] query out_features: expected=896, actual=896
[PASS] Layer 0 (target) Q weights [0:320] match source
[PASS] Layer 0 (target) K weights match
[PASS] Layer 0 (target) V weights match
[PASS] Layer 0 (target) Q bias match
[PASS] Layer 0 (target) W_O columns match
[PASS] Layer 1 Q weights [320:576] match source
[PASS] Layer 1 K weights match
[PASS] Layer 1 V weights match
[PASS] Layer 1 Q bias match
[PAS

True

In [67]:
from merge_helper import merge_bert_layers

In [69]:
for merge_layer, layer in enumerate(layer_composition):
    if len(layer) > 1:
        print("Merged Layer:", merge_layer)
        for _ in range(len(layer) -  1):
            merged = merge_bert_layers(model, merge_layer, merge_layer + 1)
            model.bert.encoder.layer[merge_layer] = merged
            del model.bert.encoder.layer[merge_layer + 1]

Merged Layer: 1
Merged Layer: 4
Merged Layer: 5


In [70]:
model.to(device)

device

device(type='cuda', index=1)

In [71]:
eval_loop(model, eval_dataloader, task_name, device)

({'accuracy': 0.33507246376811595, 'f1': 0.0}, 1.0887177597593378)

In [72]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

# 8. Learning Rate and Leanring Scheduler Setup

In [75]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

In [76]:
from train_eval_func import set_lr_scheduler

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = set_lr_scheduler(
    optimizer          = optimizer,
    num_training_steps = num_training_steps
)

print(num_training_steps)

1150


In [77]:
patience=3

In [80]:
student_layer_ids = [1,2,3,4,5,6]

In [81]:
teacher_layer_ids = []
for layer_idx in layer_composition:
    teacher_layer_ids.append(layer_idx[-1])

teacher_layer_ids

[0, 2, 3, 4, 7, 11]

In [82]:
layer_composition

[[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]

# 8. Training

In [83]:
from train_eval_func import train

In [84]:
train_stats = train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    num_epochs=num_epochs,
    lr=lr_scheduler,
    patience=patience,
    use_early_stopping=True,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    # save_path=save_path,
    display_epoch_iter=True,
    teacher_model=teacher_model,
    student_layer_ids=student_layer_ids,
    teacher_layer_ids=student_layer_ids
)

Training:   0%|          | 0/1150 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/scratch_train/../train_eval_func.py:240: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|█         | 115/1150 [00:16<02:11,  7.87it/s]


<----------------- Epoch 1 ----------------->
Loss: 2.77, Training Metrics:
  accuracy: 0.6985
  f1: 0.8059
Validation Loss: 0.62, Validation Metrics:
  accuracy: 0.7770
  f1: 0.8496
Elapsed Time: 17.2059 sec


Training:  20%|██        | 230/1150 [00:33<01:58,  7.77it/s]

...


<----------------- Epoch 8 ----------------->
Loss: 0.17, Training Metrics:
  accuracy: 0.9869
  f1: 0.9903
Validation Loss: 0.96, Validation Metrics:
  accuracy: 0.8554
  f1: 0.8995
Elapsed Time: 17.9164 sec


Training:  90%|█████████ | 1035/1150 [02:38<00:15,  7.26it/s]


<----------------- Epoch 9 ----------------->
Loss: 0.15, Training Metrics:
  accuracy: 0.9869
  f1: 0.9903
Validation Loss: 0.95, Validation Metrics:
  accuracy: 0.8529
  f1: 0.8973
Elapsed Time: 17.8642 sec


Training: 100%|██████████| 1150/1150 [02:56<00:00,  7.47it/s]


<----------------- Epoch 10 ----------------->
Loss: 0.13, Training Metrics:
  accuracy: 0.9902
  f1: 0.9927
Validation Loss: 0.98, Validation Metrics:
  accuracy: 0.8480
  f1: 0.8953
Elapsed Time: 17.9096 sec

Total Training Time: 177.27 sec


In [85]:
eval_loop(model, eval_dataloader, task_name, device)

Training: 100%|██████████| 1150/1150 [02:57<00:00,  6.49it/s]


({'accuracy': 0.8133333333333334, 'f1': 0.8674897119341564},
 1.1456258710887697)

In [86]:
eval_loop(teacher_model, eval_dataloader, task_name, device)

({'accuracy': 0.8133333333333334, 'f1': 0.8686786296900489},
 1.1832497396411512)